# Comprehensive Evaluation

Compares all four methods against the MinView geological reference. Run the four
method notebooks first (they write to `outputs/`).

**What changed vs the original**
- **Cohen's kappa is now computed correctly.** The old code compared the 3 true
  classes against raw cluster ids (0..k-1) directly, which is meaningless and is
  why the paper's kappas were ~0 / negative. Each cluster is now mapped to the
  ground-truth class it most overlaps (majority vote) before OA/kappa.
- Adds **Overall Accuracy**, **Balanced Accuracy** (macro-recall - treats the
  small Kanimblan class equally, countering the "aggregate metrics mislead"
  problem), and **per-class Kanimblan recall** (the exploration-critical number).
- Intrinsic metrics (Silhouette/CH/DB) are read from each method's own feature
  space (computed in its notebook), instead of SOM winner coordinates.
- All four methods are compared on **identically majority-filtered** maps.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import rioxarray as rxr
from src import config
from src.metrics import external_metrics

geology = rxr.open_rasterio(str(config.GEOLOGY_REF_TIF)).squeeze()
extent = [float(geology.x.min()), float(geology.x.max()),
          float(geology.y.min()), float(geology.y.max())]
ref = geology.values.flatten()
print("Reference classes:", {int(k): v for k, v in config.CLASS_NAMES.items()})

In [ ]:
METHODS = {"SOM (9 clusters)": "som", "k-means + PCA": "pca",
           "k-means + CA": "ca", "k-means + SA": "sa"}

# Only score pixels where the prediction is valid AND the reference is a real
# geological class (the raster uses -9999 for no-data, which must be excluded).
VALID_CLASSES = list(config.CLASS_NAMES.keys())

maps, rows = {}, {}
for label, key in METHODS.items():
    p = config.OUTPUTS_DIR / f"{key}_labels_spatial.npy"
    if not p.exists():
        print(f"  MISSING {p.name} - run its notebook first"); continue
    m = np.load(p); maps[label] = m
    pred = m.flatten()
    mask = ~np.isnan(pred) & np.isin(ref, VALID_CLASSES)
    ext = external_metrics(ref[mask].astype(int), pred[mask].astype(int))
    # merge intrinsic metrics computed in the method's own feature space
    ij = config.OUTPUTS_DIR / f"{key}_intrinsic.json"
    if ij.exists():
        ext.update(json.load(open(ij)))
    rows[label] = ext
    print(f"{label:20s}  OA={ext['Overall Accuracy']:.3f}  "
          f"Balanced={ext['Balanced Accuracy']:.3f}  Kappa={ext['Kappa']:.3f}  "
          f"Kanimblan recall={ext['Kanimblan Recall']:.3f}")

In [ ]:
# Summary table (public metrics only).
show = ["Overall Accuracy", "Balanced Accuracy", "Kappa", "Kanimblan Recall",
        "ARI", "NMI", "Purity", "Homogeneity", "Completeness", "V-measure",
        "Silhouette", "CH Index", "DB Index", "n_clusters"]
df = pd.DataFrame(rows).T
df = df[[c for c in show if c in df.columns]].astype(float).round(4)
config.FIGURES_DIR.mkdir(exist_ok=True)
df.to_csv(config.FIGURES_DIR / "model_comparison_results.csv")
print(df.to_string())

In [ ]:
# Heatmap of the comparable [0,1] metrics.
hm_cols = ["Overall Accuracy", "Balanced Accuracy", "Kappa", "Kanimblan Recall",
           "ARI", "NMI", "Purity", "V-measure"]
hm = df[[c for c in hm_cols if c in df.columns]]
plt.figure(figsize=(11, 4))
sns.heatmap(hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1,
            linewidths=1, linecolor="white", cbar_kws={"label": "Score"})
plt.title("Model comparison (higher = better)"); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "model_comparison_heatmap.png", dpi=200); plt.show()

In [ ]:
# Side-by-side classification maps.
n = len(maps) + 1; ncol = 3; nrow = (n + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(20, 6*nrow)); axes = axes.ravel()
im = axes[0].imshow(geology.values, extent=extent, cmap="tab10", vmin=1, vmax=3, interpolation="nearest")
axes[0].set_title("Reference geology (Kanimblan/GAB/CSP)")
cb = plt.colorbar(im, ax=axes[0], ticks=[1,2,3], fraction=0.046); cb.set_ticklabels(["Kanimblan","GAB","CSP"])
for ax, (label, m) in zip(axes[1:], maps.items()):
    nc = int(np.nanmax(m))+1
    cmap = plt.get_cmap("tab10", nc).copy(); cmap.set_bad("white")
    ax.imshow(m, extent=extent, cmap=cmap, interpolation="nearest")
    ax.set_title(f"{label}\nOA={rows[label]['Overall Accuracy']:.3f} | Kappa={rows[label]['Kappa']:.3f}")
for ax in axes[n:]: ax.axis("off")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "model_comparison_maps.png", dpi=200); plt.show()

In [ ]:
# Confusion matrices (rows = true unit, cols = predicted cluster), row-normalised.
from sklearn.metrics import confusion_matrix
fig, axes = plt.subplots(2, 2, figsize=(14, 11)); axes = axes.ravel()
for ax, (label, m) in zip(axes, maps.items()):
    pred = m.flatten(); mask = ~np.isnan(pred) & np.isin(ref, VALID_CLASSES)
    yt = ref[mask].astype(int); yp = pred[mask].astype(int)
    cm = confusion_matrix(yt, yp)
    cmn = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", ax=ax,
                yticklabels=[config.CLASS_NAMES[c] for c in sorted(np.unique(yt))],
                xticklabels=sorted(np.unique(yp)))
    ax.set_title(f"{label}"); ax.set_xlabel("Predicted cluster"); ax.set_ylabel("True unit")
for ax in axes[len(maps):]: ax.axis("off")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "confusion_matrices.png", dpi=200); plt.show()